<a href="https://colab.research.google.com/github/acamogh/RAG_QUIZ-APP/blob/main/RAG_Quiz_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfplumber -q
import pdfplumber
import re
import json
from collections import Counter
from pathlib import Path

def detect_category(text):
    """Refined category detection logic."""
    t = text.lower()
    if any(k in t for k in ["google cloud", "gcp", "vertex ai", "gemini"]):
        return "Google Cloud's gen AI offerings"
    if any(k in t for k in ["fundamental", "concept", "transformer", "llm"]):
        return "Fundamentals of gen AI"
    if any(k in t for k in ["strategy", "roi", "governance", "adoption", "business"]):
        return "Business strategies for a successful gen AI solution"
    if any(k in t for k in ["prompt", "improve", "fine-tune", "technique", "output"]):
        return "Techniques to improve gen AI model output"
    return "Fundamentals of gen AI" # Fallback

def extract_questions(pdf_path):
    questions = []
    with pdfplumber.open(pdf_path) as pdf:
        # Create a mapping of page number to text
        pages_content = {i+1: p.extract_text() or "" for i, p in enumerate(pdf.pages)}
        full_text = "\n".join(pages_content.values())

    # Improved Regex: Looks for "Question X" or "Q X"
    pattern = re.compile(r'(Question\s+\d+|Q\d+)[\s\.:]', re.IGNORECASE)
    splits = pattern.split(full_text)

    # Reconstruct blocks
    question_blocks = []
    for i in range(1, len(splits), 2):
        block = (splits[i] + splits[i+1]).strip()
        if len(block) > 50: # Filter out headers/footers
            question_blocks.append(block)

    for block in question_blocks:
        # Determine Page Number
        page_num = 1
        for p, content in pages_content.items():
            if block[:50] in content:
                page_num = p
                break

        questions.append({
            "question_id": len(questions) + 1,
            "question": block.split('\n')[0].strip(),
            "full_text": block,
            "category": detect_category(block),
            "page_number": page_num,
            "source_pdf": pdf_path.name
        })
    return questions

# ====================== RUN ======================
all_questions = []
for pdf_file in list(Path(".").glob("*.[pP][dD][fF]")):
    all_questions.extend(extract_questions(pdf_file))

with open("processed_questions.json", "w", encoding="utf-8") as f:
    json.dump(all_questions, f, indent=2)

print(f"✅ Processed {len(all_questions)} questions.")

In [ ]:
# ========================= STEP 2: Upload to Pinecone =========================


!pip install pinecone sentence-transformers -q

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import json
from google.colab import userdata

# 3. Get Pinecone API Key from Colab Secrets
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# 4. Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "genai-exam-questions"   # Name of your vector database

# 5. Create index if it doesn't exist
if index_name not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,                    # Size of each vector
        metric="cosine",                  # How to measure similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("New index created")
else:
    print("Connected to existing index")

# 6. Connect to the index
index = pc.Index(index_name)

# 7. Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 8. Load questions from Step 1
with open("processed_questions.json", "r", encoding="utf-8") as f:
    all_questions = json.load(f)

print(f"Starting upload of {len(all_questions)} questions...")

# 9. Convert questions into vectors
vectors = []
for i, q in enumerate(all_questions):
    text = q.get("full_text", q.get("question", ""))   # Use full text for better search
    embedding = embedder.encode(text).tolist()         # Convert text → numbers
    vectors.append({
        "id": f"q_{i}",                                # Unique ID
        "values": embedding,                           # The vector (numbers)
        "metadata": {                                  # Extra information
            "question": text[:500],
            "full_text": text,
            "category": q.get("category", "Uncategorized"),
            "page_number": q.get("page_number", 1),
            "source_pdf": q.get("source_pdf", "")
        }
    })

# 10. Upload to Pinecone in batches
batch_size = 40
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i:i+batch_size])   # Send batch to Pinecone

print("✅ Step 2 Completed - All questions are now in Pinecone!")

# 11. Show statistics
stats = index.describe_index_stats()
print(stats)

In [ ]:
import gradio as gr
import json
import random
import os
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata

# --- SETUP ---
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("genai-exam-questions")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
groq_client = Groq(api_key=GROQ_API_KEY)

current_data = None
current_category = "Fundamentals of gen AI"
DB_FILE = "revision_db.json"
VALID_CATEGORIES = [
    "Google Cloud's gen AI offerings",
    "Fundamentals of gen AI",
    "Business strategies for a successful gen AI solution",
    "Techniques to improve gen AI model output"
]
CLUSTER_TAXONOMY = {
    "Google Cloud's gen AI offerings": ["Model Selection", "Deployment & Scaling", "API Integration", "Pricing & Quotas"],
    "Fundamentals of gen AI": ["LLM Architecture", "Embeddings & Vectorization", "Training vs Inference", "Bias & Ethics"],
    "Business strategies for a successful gen AI solution": ["ROI & Cost Optimization", "Data Governance", "Use Case Identification", "Change Management"],
    "Techniques to improve gen AI model output": ["Prompt Engineering", "Fine-tuning", "RAG (Retrieval Augmented Gen)", "Evaluation Metrics"]
}

# --- DIAGNOSTIC AGENT ---
def diagnose_and_log(question, user_ans, correct_ans, explanation, category):


    if not os.path.exists(DB_FILE):
        with open(DB_FILE, "w") as f: json.dump({}, f)
    with open(DB_FILE, "r") as f: db = json.load(f)

    target_category = category if category in VALID_CATEGORIES else "Fundamentals of gen AI"
    allowed_clusters = CLUSTER_TAXONOMY.get(target_category, ["General Concepts"])

    if target_category not in db: db[target_category] = {}

    prompt = f"""
    Analyze this incorrect exam answer:
    Question: {question}
    Category: {target_category}
    Options to choose from: {allowed_clusters}

    TASK: Classify this error into EXACTLY ONE of the provided options above.
    Return ONLY valid JSON: {{"cluster": "..."}}
    """
    try:
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        res = json.loads(response.choices[0].message.content.replace("```json", "").replace("```", ""))
        cluster = res.get('cluster', 'General Concepts')
    except:
        cluster = "General Concepts"

    if cluster not in db[target_category]:
        db[target_category][cluster] = {"occurrence": 0, "failures": []}

    db[target_category][cluster]["occurrence"] += 1
    db[target_category][cluster]["failures"].append({"question": question, "explanation": explanation})

    with open(DB_FILE, "w") as f: json.dump(db, f, indent=2)
    return {"cluster": cluster}

# --- CORE FUNCTIONS ---
def get_question_from_groq(category="All"):
    global current_data, current_category
    current_category = category if category != "All" else random.choice(VALID_CATEGORIES)
    try:
        filter_dict = {"category": current_category}
        queries = ["generative ai", "google cloud AI", "gen AI business strategy", "prompt engineering techniques"]
        query = random.choice(queries)
        result = index.query(vector=embedder.encode(query).tolist(), top_k=5, filter=filter_dict, include_metadata=True)
        if not result['matches']: return "No content found.", gr.update(choices=[]), "**Error:** No content."

        match = random.choice(result['matches'])
        meta = match['metadata']
        current_category = meta.get('category', current_category)

        context = meta.get('full_text', meta.get('question', ''))
        prompt = f"Create one challenging MCQ based on: {context}\nReturn ONLY valid JSON: {{\"question\": \"...\", \"options\": {{\"A\": \"...\", \"B\": \"...\", \"C\": \"...\", \"D\": \"...\"}}, \"correct_answer\": \"A\", \"explanation\": \"...\"}}"
        response = groq_client.chat.completions.create(model="openai/gpt-oss-120b", messages=[{"role": "user", "content": prompt}])
        data = json.loads(response.choices[0].message.content.replace("```json", "").replace("```", ""))
        current_data = data
        options = [f"{k}. {v}" for k, v in data["options"].items()]
        source = f"📖 **Source:** {meta.get('source_pdf', 'N/A')} | **Category:** {current_category}"
        return data["question"], gr.update(choices=options), source
    except Exception as e: return f"Error: {str(e)}", gr.update(choices=[]), ""

def check_answer(selected_option):
    global current_data, current_category
    if not current_data or not selected_option: return "Please load and answer.", ""

    correct_key = current_data.get("correct_answer", "A")
    correct_text = current_data.get("options", {}).get(correct_key, "Unknown")
    user_choice = selected_option[0].upper()
    explanation = current_data.get('explanation', 'No explanation provided.')

    if user_choice == correct_key:
        return "✅ **Correct!**", f"**Explanation:**\n{explanation}"
    else:
        diag = diagnose_and_log(current_data['question'], user_choice, correct_key, explanation, current_category)
        feedback = f"❌ **Wrong!** The correct answer was **{correct_key}: {correct_text}**"
        summary = f"**Diagnostic Cluster:** {diag['cluster']}\n\n**Explanation:**\n{explanation}"
        return feedback, summary

# --- UI ---
with gr.Blocks(title="Gen AI Exam") as demo:
    gr.Markdown("# 🧠 Google Cloud Gen AI Exam Simulator")

    with gr.Tab("📋 Full Exam"):
        load_btn = gr.Button("Load Question", variant="primary")
        source_display = gr.Markdown()
        question_box = gr.Textbox(label="Question", lines=6)
        options_radio = gr.Radio(choices=[], label="Answer")
        submit_btn = gr.Button("Submit")
        feedback_box = gr.Markdown()
        explanation_box = gr.Markdown()
        load_btn.click(get_question_from_groq, outputs=[question_box, options_radio, source_display])
        submit_btn.click(check_answer, inputs=options_radio, outputs=[feedback_box, explanation_box])

    with gr.Tab("🔍 Category Practice"):
        category_drop = gr.Dropdown(choices=["All"] + VALID_CATEGORIES, value="All", label="Select Category")
        load_btn_cat = gr.Button("Load Question")
        source_display_cat = gr.Markdown()
        question_box_cat = gr.Textbox(label="Question", lines=6)
        options_radio_cat = gr.Radio(choices=[], label="Answer")
        submit_btn_cat = gr.Button("Submit")
        feedback_box_cat = gr.Markdown()
        explanation_box_cat = gr.Markdown()
        load_btn_cat.click(get_question_from_groq, inputs=category_drop, outputs=[question_box_cat, options_radio_cat, source_display_cat])
        submit_btn_cat.click(check_answer, inputs=options_radio_cat, outputs=[feedback_box_cat, explanation_box_cat])

    with gr.Tab("📚 Smart Revision"):
        rev_box = gr.JSON(label="Revision Database")
        gr.Button("Refresh").click(lambda: json.load(open(DB_FILE)) if os.path.exists(DB_FILE) else {}, outputs=rev_box)

demo.launch(share=True, server_port=7860)

In [ ]:
!fuser -k 7860/tcp
!fuser -k 7861/tcp